In [29]:
#Product demand analysis
#Obj is to analyse product level sales contribution, demand stability and concenteation of revenue to support business decision making 

import pandas as pd
df = pd.read_csv("data/store_sales_long.csv")
df.head()


,item,store_id,month,unit_sales
0,A,1,2023-01-01,0.0
1,A,2,2023-01-01,5.0
2,A,3,2023-01-01,5.0
3,A,4,2023-01-01,20.0
4,A,5,2023-01-01,0.0


In [30]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113253 entries, 0 to 113252
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   item        113253 non-null  object 
 1   store_id    113253 non-null  int64  
 2   month       113253 non-null  object 
 3   unit_sales  113253 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 3.5+ MB


In [31]:
df["month"]= pd.to_datetime(df["month"])

In [32]:
product_monthly=(
    df.groupby(["item","month"],as_index=False)["unit_sales"]
    .sum()
)

In [33]:
product_monthly.head()

,item,month,unit_sales
0,A,2022-01-01,4162.0
1,A,2022-02-01,3966.0
2,A,2022-03-01,4438.0
3,A,2022-04-01,14128.0
4,A,2022-05-01,10814.0


In [34]:

#seriesgroupby
volatility=(
    product_monthly
    .groupby("item")["unit_sales"]
    .agg(
        avg_monthly_sales="mean",
        std_monthly_sales="std"
    )
    .reset_index()
)

volatility

,item,avg_monthly_sales,std_monthly_sales
0,A,7881.904762,7088.345307
1,B,7617.190476,7590.876442
2,C,5913.095238,5441.922977
3,D,3761.285714,3571.065529
4,E,452.809524,396.677403
5,F,488.238095,553.151327
6,G,1159.380952,1958.123093
7,H,239.333333,258.815249


In [35]:
#dataframegroupby
volatility =(
    product_monthly
    .groupby("item",as_index=False)
    .agg(
        avg_monthly_sales=("unit_sales", "mean"),
        std_monthly_sales=("unit_sales","std")
    )
)

volatility

,item,avg_monthly_sales,std_monthly_sales
0,A,7881.904762,7088.345307
1,B,7617.190476,7590.876442
2,C,5913.095238,5441.922977
3,D,3761.285714,3571.065529
4,E,452.809524,396.677403
5,F,488.238095,553.151327
6,G,1159.380952,1958.123093
7,H,239.333333,258.815249


In [36]:
#coeffecient of variation ie Volatility score
volatility["cv"]= volatility["std_monthly_sales"]/volatility["avg_monthly_sales"]

In [37]:
volatility.sort_values("cv",ascending=False).head(10)


,item,avg_monthly_sales,std_monthly_sales,cv
6,G,1159.380952,1958.123093,1.688938
5,F,488.238095,553.151327,1.132954
7,H,239.333333,258.815249,1.081401
1,B,7617.190476,7590.876442,0.996545
3,D,3761.285714,3571.065529,0.949427
2,C,5913.095238,5441.922977,0.920317
0,A,7881.904762,7088.345307,0.899319
4,E,452.809524,396.677403,0.876036


In [38]:
volatility.sort_values("cv", ascending=True).head(10)

,item,avg_monthly_sales,std_monthly_sales,cv
4,E,452.809524,396.677403,0.876036
0,A,7881.904762,7088.345307,0.899319
2,C,5913.095238,5441.922977,0.920317
3,D,3761.285714,3571.065529,0.949427
1,B,7617.190476,7590.876442,0.996545
7,H,239.333333,258.815249,1.081401
5,F,488.238095,553.151327,1.132954
6,G,1159.380952,1958.123093,1.688938


In [39]:
def stability_label(cv):
    if cv<= 0.5:
        return "Stable"
    elif cv<= 1.0:
        return "Moderate"
    else:
        return "Volatile"

volatility["stability"]= volatility["cv"].apply(stability_label)

volatility


,item,avg_monthly_sales,std_monthly_sales,cv,stability
0,A,7881.904762,7088.345307,0.899319,Moderate
1,B,7617.190476,7590.876442,0.996545,Moderate
2,C,5913.095238,5441.922977,0.920317,Moderate
3,D,3761.285714,3571.065529,0.949427,Moderate
4,E,452.809524,396.677403,0.876036,Moderate
5,F,488.238095,553.151327,1.132954,Volatile
6,G,1159.380952,1958.123093,1.688938,Volatile
7,H,239.333333,258.815249,1.081401,Volatile


In [40]:
#ABC/Pareto analysis
#Step A Product level total sales
product_sales = (
    df.groupby("item", as_index=False)["unit_sales"]
    .sum()
    .rename(columns={"unit_sales":"total_sales"})
)
product_sales.sort_values("total_sales",ascending=False)


,item,total_sales
0,A,165520.0
1,B,159961.0
2,C,124175.0
3,D,78987.0
6,G,24347.0
5,F,10253.0
4,E,9509.0
7,H,5026.0


In [41]:
#Contribution percentage
total_sales_all = product_sales["total_sales"].sum()
product_sales["sales_share_pct"]=(
    product_sales["total_sales"]/total_sales_all * 100
)
product_sales.sort_values("sales_share_pct",ascending=False)

,item,total_sales,sales_share_pct
0,A,165520.0,28.647681
1,B,159961.0,27.685547
2,C,124175.0,21.491819
3,D,78987.0,13.670822
6,G,24347.0,4.213902
5,F,10253.0,1.774557
4,E,9509.0,1.645788
7,H,5026.0,0.869884


In [42]:
product_sales=product_sales.sort_values("sales_share_pct", ascending=False)

product_sales["cumulative_share_pct"]=(
    product_sales["sales_share_pct"].cumsum()
)

product_sales

,item,total_sales,sales_share_pct,cumulative_share_pct
0,A,165520.0,28.647681,28.647681
1,B,159961.0,27.685547,56.333228
2,C,124175.0,21.491819,77.825047
3,D,78987.0,13.670822,91.495869
6,G,24347.0,4.213902,95.709771
5,F,10253.0,1.774557,97.484328
4,E,9509.0,1.645788,99.130116
7,H,5026.0,0.869884,100.000000


In [43]:
#ABC classification ie formally labelling

def abc_label(cum_pct):
    if cum_pct<=80:
        return "A"
    elif cum_pct<=95:
        return "B"
    else:
        return "C"

product_sales["ABC_class"]=product_sales["cumulative_share_pct"].apply(abc_label)

product_sales

,item,total_sales,sales_share_pct,cumulative_share_pct,ABC_class
0,A,165520.0,28.647681,28.647681,A
1,B,159961.0,27.685547,56.333228,A
2,C,124175.0,21.491819,77.825047,A
3,D,78987.0,13.670822,91.495869,B
6,G,24347.0,4.213902,95.709771,C
5,F,10253.0,1.774557,97.484328,C
4,E,9509.0,1.645788,99.130116,C
7,H,5026.0,0.869884,100.000000,C


In [44]:
final_product_analysis= product_sales.merge(
    volatility[["item","avg_monthly_sales","std_monthly_sales","cv", "stability"]],on="item", how="left"
)

final_product_analysis

,item,total_sales,sales_share_pct,cumulative_share_pct,ABC_class,avg_monthly_sales,std_monthly_sales,cv,stability
0,A,165520.0,28.647681,28.647681,A,7881.904762,7088.345307,0.899319,Moderate
1,B,159961.0,27.685547,56.333228,A,7617.190476,7590.876442,0.996545,Moderate
2,C,124175.0,21.491819,77.825047,A,5913.095238,5441.922977,0.920317,Moderate
3,D,78987.0,13.670822,91.495869,B,3761.285714,3571.065529,0.949427,Moderate
4,G,24347.0,4.213902,95.709771,C,1159.380952,1958.123093,1.688938,Volatile
5,F,10253.0,1.774557,97.484328,C,488.238095,553.151327,1.132954,Volatile
6,E,9509.0,1.645788,99.130116,C,452.809524,396.677403,0.876036,Moderate
7,H,5026.0,0.869884,100.000000,C,239.333333,258.815249,1.081401,Volatile


In [45]:
final_product_analysis["segment"]=(
    final_product_analysis["ABC_class"]+"-"+ final_product_analysis["stability"]
)

final_product_analysis

,item,total_sales,sales_share_pct,cumulative_share_pct,ABC_class,avg_monthly_sales,std_monthly_sales,cv,stability,segment
0,A,165520.0,28.647681,28.647681,A,7881.904762,7088.345307,0.899319,Moderate,A-Moderate
1,B,159961.0,27.685547,56.333228,A,7617.190476,7590.876442,0.996545,Moderate,A-Moderate
2,C,124175.0,21.491819,77.825047,A,5913.095238,5441.922977,0.920317,Moderate,A-Moderate
3,D,78987.0,13.670822,91.495869,B,3761.285714,3571.065529,0.949427,Moderate,B-Moderate
4,G,24347.0,4.213902,95.709771,C,1159.380952,1958.123093,1.688938,Volatile,C-Volatile
5,F,10253.0,1.774557,97.484328,C,488.238095,553.151327,1.132954,Volatile,C-Volatile
6,E,9509.0,1.645788,99.130116,C,452.809524,396.677403,0.876036,Moderate,C-Moderate
7,H,5026.0,0.869884,100.000000,C,239.333333,258.815249,1.081401,Volatile,C-Volatile


In [46]:
final_product_analysis.sort_values(
    ["ABC_class", "cv"],
    ascending=[True, False]
)

,item,total_sales,sales_share_pct,cumulative_share_pct,ABC_class,avg_monthly_sales,std_monthly_sales,cv,stability,segment
1,B,159961.0,27.685547,56.333228,A,7617.190476,7590.876442,0.996545,Moderate,A-Moderate
2,C,124175.0,21.491819,77.825047,A,5913.095238,5441.922977,0.920317,Moderate,A-Moderate
0,A,165520.0,28.647681,28.647681,A,7881.904762,7088.345307,0.899319,Moderate,A-Moderate
3,D,78987.0,13.670822,91.495869,B,3761.285714,3571.065529,0.949427,Moderate,B-Moderate
4,G,24347.0,4.213902,95.709771,C,1159.380952,1958.123093,1.688938,Volatile,C-Volatile
5,F,10253.0,1.774557,97.484328,C,488.238095,553.151327,1.132954,Volatile,C-Volatile
7,H,5026.0,0.869884,100.000000,C,239.333333,258.815249,1.081401,Volatile,C-Volatile
6,E,9509.0,1.645788,99.130116,C,452.809524,396.677403,0.876036,Moderate,C-Moderate


In [47]:
final_product_analysis.to_csv("data/product_portfolio_analysis.csv", index=False)

In [49]:
final_product_analysis.to_csv("C:\Data analyst projects\stock sales data 20222023\data\processed\product_portfolio_analysis.csv",index=False)